<center style="font-size:2em;">Список ключевых аннотаций</center>

<center style="font-size:2em;">@Inject</center>

Аннотация вставляет свой код внутрь целевого метода `method` в место, задаваемое в параметре `at` с помощью указателя на инструкцию байткода :

| `value`      | Действие                                            | Позиция вставки |
| :----------- | :-------------------------------------------------- | :-------------- |
| `RETURN`     | ищет инструкцию возврата значения                   | **перед**       |
| `TAIL`       | ищет конец метода (последняя инструкция)            | **перед**       |
| `HEAD`       | ищет начало метода                                  | **после**       |
| `INVOKE`     | ищет вызов метода                                   | **перед/после** |
| `FIELD`      | ищет доступ к полю                                  | **перед/после** |
| `NEW`        | ищет создание нового объекта                        | **перед/после** |
| `EXCEPTION`  | ищет начало блока `catch`                           | **перед**       |
| `JUMP`       | ищет инструкцию перехода                            | **перед**       |
| `INVOKE_STRING` | ищет вызов метода с единственным параметром `String` и возвращаемым типом `void`                                                               | **перед** |
| `INVOKE_ASSIGN` | ищет вызов метода с присваиванием результата     | **после**       |

**Дополнительные параметры `@At` :**
- `shift = At.Shift.BEFORE/AFTER` - явно указывает позицию вставки
- `ordinal` - индекс найденной инструкции (начиная с 0)
- `target` - точная сигнатура метода/поля
- `opcode` - конкретный опкод для `FIELD` (например, `GETFIELD`, `PUTFIELD`)

**Дополнительные параметры `@Inject` :**
- `cancellable = true` - позволяет настроить параметр `CallbackInfo ci` для `void` методов и `CallbackInfoReturnable<T> cir` для методов с возвращаемым значением, где `T` - тип возвращаемого значения
- `locals = LocalCapture.CAPTURE_FAILSOFT/CAPTURE_FAILHARD/CAPTURE_FAILEXCEPTION` - позволяет захватывать локальные переменные

**Типы callback параметров :**
- `cancellable = false` - просто выполняет код
- `CallbackInfo ci` - может прервать выполнение (`ci.cancel()`)
- `CallbackInfoReturnable<T> cir` - может вернуть значение (`cir.setReturnValue()`)

-------------------------------------------------------------------------------------

Реализация вставки кода перед вызовом конкретного метода :
```java
@Inject(method = "target()V", at = @At(value = "INVOKE", target = "Lnet/example/Dummy;dummy()V"))
private void mixin(CallbackInfo ci) {
    injectedCode();
}
```
Изменённый оригинальный класс, где строка с `+` обозначает вставленный код :
```java
public void target() {
+   injectedCode();
    Dummy.getInstance().dummy();
}
```

-------------------------------------------------------------------------------

Реализация с использованием параметра `cancellable = true` :
```java
@Inject(method = "target()V", 
        at = @At(value = "INVOKE", 
                 target = "Lnet/example/Dummy;dummy()V"), 
                 cancellable = true)
private void mixin(CallbackInfo ci) {
    if (condition) {
        ci.cancel();
    }
}
```
Изменённый оригинальный класс, где строка с `+` обозначает вставленный код :
```java
public void target() {
    Dummy.getInstance().dummy();
+   {
+       boolean canceled = false;
+       if (condition) {
+           canceled = true;
+       }
+       if (canceled) return;
+   }
}
```

-------------------------------------------------------------------------------

Реализация с использованием параметра `locals = LocalCapture.CAPTURE_FAILSOFT` для захвата локальных переменных :
```java
@Inject(method = "target()V", at = @At(value = "INVOKE", target = "Lnet/example/Dummy;dummy()V"), locals = LocalCapture.CAPTURE_FAILSOFT)
private void mixin(CallbackInfo ci, int i, int b) {
    if (i < b) {
        System.out.println(i + "<" + "b");
    }
}
```
Изменённый оригинальный класс, где строка с `+` обозначает вставленный код :
```java
public void target() {
  	int i = 0;
    int b = 1;
    Dummy.getInstance().dummy();
+   {
+       if (i < b) {
+           System.out.println(i + "<" + "b");
+       }
+   }
}
```

-------------------------------------------------------------------------------------


<center style="font-size:2em;">@Redirect</center>

Аннотация **полностью заменяет** обращение к целевому объекту указанному в `at` параметре из метода указанного в `method` параметре на вызов своего метода :

| `value`      | Действие                                            | 
| :----------- | :-------------------------------------------------- |
| `INVOKE`     | заменяет вызов метода на свой метод                 |
| `FIELD`      | заменяет обращение к полю на свой метод             |
| `NEW`        | заменяет создание объекта на свой метод             |
| `INSTANCEOF` | заменяет проверку типа на свой метод                |

**Дополнительные параметры `@At` :**
- `ordinal` - индекс найденной инструкции (начиная с 0)
- `target` - точная сигнатура метода/поля/класса/класса
- `opcode` - конкретный опкод для `FIELD` (например, `GETFIELD`, `PUTFIELD`)

**ВАЖНО :** `@Redirect` НЕ поддерживает :
- `cancellable` - это параметр только для `@Inject`
- `locals` - локальные переменные недоступны в `@Redirect`
- `shift` - не применимо, так как метод полностью заменяется

**Сигнатуры redirect методов с параметрами:**

- Для `INVOKE` :  
    `(<TargetClass> instance, <Type1> arg1, <Type2> arg2, ...)`  
    - `instance` — объект, на котором вызывается метод (если не static)  
    - `argN` — параметры оригинального метода

- Для `FIELD` (GET):  
    `(<TargetClass> instance)`  
    - `instance` — объект, из которого читается поле (если не static)

- Для `FIELD` (PUT):  
    `(<TargetClass> instance, <FieldType> value)`  
    - `instance` — объект, в котором записывается поле (если не static)  
    - `value` — новое значение поля

- Для `NEW`:  
    `(<CtorType1> arg1, <CtorType2> arg2, ...)`  
    - `argN` — параметры конструктора создаваемого объекта

- Для `INSTANCEOF`:  
    `(Object obj)`  
    - `obj` — объект, который проверяется на принадлежность к типу

------------------------------------------------------------------------------------

Реализация в случае с `INVOKE` :  
Целевой класс :
```java
public class Player {
    private int health = 100;
    private int maxHealth = 100;
    
    public void takeDamage(int damage) {
        // Оригинальная логика - прямое вычитание урона
        this.health = Math.max(0, this.health - calculateDamage(damage));
        
        if (this.health <= 0) {
            this.die();
        }
    }
    
    private int calculateDamage(int baseDamage) {
        // Простое вычисление урона
        return baseDamage;
    }
    
    private void die() {
        System.out.println("Player died!");
    }
}
```  
Преобразующий класс :
```java
@Mixin(Player.class)
public class PlayerMixin {
    
    @Redirect(method = "takeDamage", 
              at = @At(value = "INVOKE", 
                      target = "LPlayer;calculateDamage(I)I"))
    private int redirectCalculateDamage(Player instance, int baseDamage) {
        // Новая логика - добавляем защиту и случайность
        int armor = 10; // Допустим, у игрока есть броня
        int randomFactor = (int)(Math.random() * 5); // Случайный фактор 0-4
        
        int finalDamage = Math.max(1, baseDamage - armor + randomFactor);
        System.out.println("Modified damage calculation: " + baseDamage + " -> " + finalDamage);
        
        return finalDamage;
    }
}
```
Результат :  
Теперь вместо простого `calculateDamage(damage)` будет вызываться модифицированный метод с учётом брони и случайности.

----------------------------------------------------------------------------------

Реализация в случае с `FIELD` :  
Целевой класс :
```java
public class GameConfig {
    public static final int MAX_PLAYERS = 10;
    private int currentPlayers = 0;
    
    public boolean canAddPlayer() {
        // Оригинальная проверка - прямое обращение к полю
        return currentPlayers < MAX_PLAYERS;
    }
    
    public void addPlayer() {
        if (canAddPlayer()) {
            currentPlayers++;
            System.out.println("Player added. Current: " + currentPlayers);
        }
    }
}
```
Преобразуюший класс : 
```java
@Mixin(GameConfig.class)
public class GameConfigMixin {
    
    @Redirect(method = "canAddPlayer",
              at = @At(value = "FIELD", 
                      target = "LGameConfig;MAX_PLAYERS:I",
                      opcode = Opcodes.GETSTATIC))
    private int getModifiedMaxPlayers() {
        // Динамически изменяем максимальное количество игроков
        // в зависимости от времени суток
        java.time.LocalTime now = java.time.LocalTime.now();
        
        if (now.getHour() >= 18 && now.getHour() <= 23) {
            // Вечернее время - больше игроков
            return 20;
        } else {
            // Обычное время
            return 10;
        }
    }
}
```
Результат:  
Вместо константы `MAX_PLAYERS = 10` будет использоваться динамическое значение в зависимости от времени.

---------------------------------------------------------------------------------

Реализация в случае с `NEW` :  
Целевой класс :
```java
public class ItemFactory {
    
    public Item createWeapon(String type, int damage) {
        // Оригинальное создание оружия
        Weapon weapon = new Weapon(type, damage);
        
        weapon.setDurability(100);
        return weapon;
    }
}

class Weapon implements Item {
    private String type;
    private int damage;
    private int durability;
    
    public Weapon(String type, int damage) {
        this.type = type;
        this.damage = damage;
    }
    
    public void setDurability(int durability) {
        this.durability = durability;
    }
}
```
Преобразующий класс :
```java
@Mixin(ItemFactory.class)
public class ItemFactoryMixin {
    
    @Redirect(method = "createWeapon",
              at = @At(value = "NEW", 
                      target = "LWeapon;"))
    private Weapon createEnhancedWeapon(String type, int damage) {
        // Создаем улучшенную версию оружия
        EnhancedWeapon enhancedWeapon = new EnhancedWeapon(type, damage);
        
        // Добавляем бонусы в зависимости от типа
        switch (type.toLowerCase()) {
            case "sword":
                enhancedWeapon.addEnchantment("Sharpness", 2);
                break;
            case "bow":
                enhancedWeapon.addEnchantment("Power", 1);
                break;
        }
        
        return enhancedWeapon;
    }
}

class EnhancedWeapon extends Weapon {
    private Map<String, Integer> enchantments = new HashMap<>();
    
    public EnhancedWeapon(String type, int damage) {
        super(type, damage);
    }
    
    public void addEnchantment(String name, int level) {
        enchantments.put(name, level);
        System.out.println("Added enchantment: " + name + " Level " + level);
    }
}
```
Результат:  
Вместо обычного Weapon будет создаваться EnhancedWeapon с дополнительными возможностями.

--------------------------------------------------------------------------------------

Реализация в случае с `INSTANCEOF` :  
Целевой класс :
```java
public class EntityProcessor {
    
    public void processEntity(Object entity) {
        // Оригинальная проверка типа
        if (entity instanceof Monster) {
            Monster monster = (Monster) entity;
            monster.attack();
        } else if (entity instanceof Player) {
            Player player = (Player) entity;
            player.move();
        }
    }
}

class Monster {
    public void attack() {
        System.out.println("Monster attacks!");
    }
}
```
Преобразуюший класс : 
```java
@Mixin(EntityProcessor.class)
public class EntityProcessorMixin {
    
    @Redirect(method = "processEntity",
              at = @At(value = "INSTANCEOF", 
                      target = "LMonster;"))
    private boolean isMonsterOrBoss(Object obj) {
        // Расширяем проверку - теперь боссы тоже считаются монстрами
        return obj instanceof Monster || obj instanceof Boss;
    }
}

class Boss {
    public void attack() {
        System.out.println("Boss performs powerful attack!");
    }
}
```
Результат:  
Теперь проверка `entity instanceof Monster` будет также возвращать `true` для объектов типа `Boss`.

--------------------------------------------------------------------------------

<center style="font-size:2em;">@ModifyArg</center>

Аннотация **изменяет конкретный аргумент** при вызове метода, указанного в параметре `at`. Позволяет модифицировать параметры перед их передачей в целевой метод :

**Параметры `@ModifyArg` :**
- `method` - целевой метод, в котором нужно изменить аргумент
- `at` - указывает на вызов метода (`INVOKE` или `NEW`)
- `index` - индекс аргумента для изменения (начиная с 0)

**Поддерживаемые значения `@At` :**
- `INVOKE` - изменяет аргумент при вызове метода
- `NEW` - изменяет аргумент при создании объекта

**Дополнительные параметры `@At` :**
- `target` - точная сигнатура метода/конструктора
- `ordinal` - индекс найденной инструкции (начиная с 0)

**ВАЖНО :** 
- Метод должен принимать один параметр того же типа, что и изменяемый аргумент
- Метод должен возвращать значение того же типа
- Для статических методов индексация начинается с 0, для нестатических с 1 (0 - это `this`)
- Можно изменять только один аргумент за раз
- Для изменения нескольких аргументов нужно несколько `@ModifyArg`
- При работе с `ordinal` можно различать несколько вызовов одного метода

------------------------------------------------------------------------------------

Сравнительная реализация : 
```java
@ModifyArg(
    method = "target()V",
    at = @At(value = "INVOKE", target = "Lnet/example/Dummy;dummy(IIII)V"),
    index = 1
)
private int mixin(int in) {
    if (in > 5) {
        return in + 10;
    } else {
        return in - 10;
    }
}
```
Изменённый оригинальный класс, где строка с `+` обозначает вставленный код :
```java
public void target() {
+   // (Parameters extracted from the method call to keep them in the correct order of definition)
+   int par1 = 1;
+   int par2;
+   {
+       int in = 2;
+       if (in > 5) {
+           par2 = in + 10;
+       } else {
+           par2 = in - 10;
+       }
+   }
+   int par3 = 3;
+   int par4 = 4;
+   Dummy.getInstance().dummy(par1, par2, par3, par4);
-   Dummy.getInstance().dummy(1, 2, 3, 4);
}
```

---------------------------------------------------------------------------

Реализация в случае с `INVOKE` :  
Целевой класс :
```java
public class DamageCalculator {
    
    public void dealDamage(Entity target, int damage) {
        // Оригинальная логика
        target.takeDamage(calculateFinalDamage(damage, target.getArmor()));
    }
    
    private int calculateFinalDamage(int baseDamage, int armor) {
        return Math.max(1, baseDamage - armor);
    }
}

class Entity {
    private int armor = 5;
    
    public int getArmor() {
        return armor;
    }
    
    public void takeDamage(int damage) {
        System.out.println("Taking " + damage + " damage");
    }
}
```
Преобразующий класс :
```java
@Mixin(DamageCalculator.class)
public class DamageCalculatorMixin {
    
    @ModifyArg(method = "dealDamage(LEntity;I)V",
               at = @At(value = "INVOKE", 
                        target = "LDamageCalculator;calculateFinalDamage(II)I"),
               index = 0)
    private int modifyBaseDamage(int baseDamage) {
        // Увеличиваем базовый урон на 50%
        int modifiedDamage = (int)(baseDamage * 1.5f);
        System.out.println("Modified damage: " + baseDamage + " -> " + modifiedDamage);
        return modifiedDamage;
    }
}
```
Результат :  
Теперь при вызове `calculateFinalDamage(damage, target.getArmor())` первый аргумент `damage` будет увеличен на 50%.

------------------------------------------------------------------------------------

Реализация с изменением второго аргумента :
```java
@Mixin(DamageCalculator.class)
public class DamageCalculatorMixin {
    
    @ModifyArg(method = "dealDamage(LEntity;I)V",
               at = @At(value = "INVOKE", 
                       target = "LDamageCalculator;calculateFinalDamage(II)I"),
               index = 1)
    private int modifyArmor(int armor) {
        // Уменьшаем эффективность брони вдвое
        int modifiedArmor = armor / 2;
        System.out.println("Modified armor: " + armor + " -> " + modifiedArmor);
        return modifiedArmor;
    }
}
```
Результат:  
Второй аргумент `target.getArmor()` будет уменьшен вдвое перед передачей в `calculateFinalDamage`.

------------------------------------------------------------------------------------

Пример в случае с `NEW` (изменение аргумента конструктора) :  
Целевой класс :
```java
public class WeaponFactory {
    
    public Weapon createSword(String material, int damage) {
        // Оригинальное создание меча
        Weapon sword = new Weapon("Sword", material, damage);
        
        sword.addProperty("melee", true);
        return sword;
    }
}

class Weapon {
    private String type;
    private String material;
    private int damage;
    
    public Weapon(String type, String material, int damage) {
        this.type = type;
        this.material = material;
        this.damage = damage;
    }
    
    public void addProperty(String key, Object value) {
        System.out.println("Added property: " + key + " = " + value);
    }
}
```
Преобразующий класс :
```java
@Mixin(WeaponFactory.class)
public class WeaponFactoryMixin {
    
    @ModifyArg(method = "createSword(Ljava/lang/String;I)LWeapon;",
               at = @At(value = "NEW", 
                       target = "(Ljava/lang/String;Ljava/lang/String;I)V"),
               index = 2)
    private int modifyWeaponDamage(int originalDamage) {
        // Увеличиваем урон мечей на основе материала
        if (originalDamage > 10) {
            return originalDamage + 5; // Бонус для мощного оружия
        }
        return originalDamage + 2; // Небольшой бонус
    }
    
    @ModifyArg(method = "createSword(Ljava/lang/String;I)LWeapon;",
               at = @At(value = "NEW", 
                       target = "(Ljava/lang/String;Ljava/lang/String;I)V"),
               index = 1)
    private String modifyMaterial(String material) {
        // Улучшаем материал
        switch (material.toLowerCase()) {
            case "iron":
                return "Enhanced Iron";
            case "steel":
                return "Tempered Steel";
            default:
                return "Refined " + material;
        }
    }
}
```
Результат :  
При создании `new Weapon("Sword", material, damage)` второй и третий аргументы будут модифицированы.

------------------------------------------------------------------------------------

Реализация в случае с несколькими `@ModifyArg` для одного вызова :
```java
@Mixin(GameEngine.class)
public class GameEngineMixin {
    
    @ModifyArg(method = "spawnEntity(Ljava/lang/String;DDI)V",
               at = @At(value = "INVOKE", 
                       target = "LWorld;addEntity(Ljava/lang/String;DDI)V"),
               index = 1)
    private double modifyX(double x) {
        // Смещаем все сущности по X на 10 блоков
        return x + 10.0;
    }
    
    @ModifyArg(method = "spawnEntity(Ljava/lang/String;DDI)V",
               at = @At(value = "INVOKE", 
                       target = "LWorld;addEntity(Ljava/lang/String;DDI)V"),
               index = 2)
    private double modifyY(double y) {
        // Поднимаем все сущности на 5 блоков
        return y + 5.0;
    }
}
```

-----------------------------------------------------------------

<center style="font-size:2em;">@ModifyArgs</center>

Аннотация **изменяет все аргументы одновременно** при вызове метода, указанного в параметре `at`. Позволяет модифицировать массив параметров перед их передачей в целевой метод:

**Параметры `@ModifyArgs` :**
- `method` - целевой метод, в котором нужно изменить аргументы
- `at` - указывает на вызов метода (`INVOKE` или `NEW`)

**Поддерживаемые значения `@At` :**
- `INVOKE` - изменяет аргументы при вызове метода
- `NEW` - изменяет аргументы при создании объекта

**Дополнительные параметры `@At` :**
- `target` - точная сигнатура метода/конструктора
- `ordinal` - индекс найденной инструкции (начиная с 0)

**ВАЖНО :** 
- Метод должен принимать один параметр типа `Args`
- Метод должен возвращать `void`
- Можно изменить все аргументы за один раз
- Аргументы доступны через методы `args.get(index)` и `args.set(index, value)`
- Изменения применяются ко всем аргументам одновременно

------------------------------------------------------------------------------------

Сравнительная реализация :  
```java
@ModifyArgs(
    method = "target",
    at = @At(value = "INVOKE", target = "Lnet/example/Dummy;dummy(IIII)V")
)
private void mixin(Args args) {
    args.set(0, args.get(0) - 1);
    args.set(1, 1);
    args.set(2, 0);
    args.set(3, 1);
}
```
Изменённый оригинальный класс, где строка с `+` обозначает вставленный код :  
```java
public void target() {
+   Dummy.getInstance().dummy(0, 1, 0, 1);
-   Dummy.getInstance().dummy(1, 2, 3, 4);
}
```

-----------------------------------------------------------------------

Реализация случая с `INVOKE` :  
Целевой класс :  
```java
public class CombatSystem {
    
    public void applyDamage(Entity target, int baseDamage, float multiplier, boolean isCritical) {
        // Оригинальная логика
        int finalDamage = calculateDamage(baseDamage, multiplier, isCritical);
        target.takeDamage(finalDamage);
    }
    
    private int calculateDamage(int baseDamage, float multiplier, boolean isCritical) {
        int damage = (int)(baseDamage * multiplier);
        if (isCritical) {
            damage *= 2;
        }
        return damage;
    }
}

class Entity {
    private int health = 100;
    
    public void takeDamage(int damage) {
        health -= damage;
        System.out.println("Took " + damage + " damage. Health: " + health);
    }
}
```
Преобразующий класс :  
```java
@Mixin(CombatSystem.class)
public class CombatSystemMixin {
    
    @ModifyArgs(method = "applyDamage(LEntity;IFZ)V",
                at = @At(value = "INVOKE", 
                        target = "LCombatSystem;calculateDamage(IFZ)I"))
    private void modifyDamageCalculation(Args args) {
        // Получаем текущие значения
        int baseDamage = args.get(0);
        float multiplier = args.get(1);
        boolean isCritical = args.get(2);
        
        // Применяем глобальные модификаторы
        int modifiedDamage = (int)(baseDamage * 1.2f); // Увеличиваем базовый урон на 20%
        float modifiedMultiplier = multiplier + 0.5f;  // Добавляем бонус к множителю
        boolean forceCritical = isCritical || (Math.random() < 0.1); // 10% шанс критического удара
        
        // Устанавливаем новые значения
        args.set(0, modifiedDamage);
        args.set(1, modifiedMultiplier);
        args.set(2, forceCritical);
        
        System.out.println("Modified damage args: " + baseDamage + "->" + modifiedDamage + 
                          ", multiplier: " + multiplier + "->" + modifiedMultiplier + 
                          ", critical: " + isCritical + "->" + forceCritical);
    }
}
```
Результат :  
Теперь при вызове `calculateDamage(baseDamage, multiplier, isCritical)` все три аргумента будут модифицированы одновременно.  

------------------------------------------------------------------------------------

Реализация случая с условной модификацией аргументов :  
```java
@Mixin(GameEngine.class)
public class GameEngineMixin {
    
    @ModifyArgs(method = "spawnEntity(Ljava/lang/String;DDIF)V",
                at = @At(value = "INVOKE", 
                        target = "LWorld;createEntity(Ljava/lang/String;DDIF)LEntity;"))
    private void modifyEntitySpawn(Args args) {
        String entityType = args.get(0);
        double x = args.get(1);
        double y = args.get(2);
        int level = args.get(3);
        float scale = args.get(4);
        
        // Особая логика для разных типов сущностей
        switch (entityType.toLowerCase()) {
            case "boss":
                // Боссы появляются в центре карты с увеличенным размером
                args.set(1, 0.0); // x = 0
                args.set(2, 0.0); // y = 0
                args.set(3, level + 5); // уровень +5
                args.set(4, scale * 2.0f); // размер x2
                break;
                
            case "treasure":
                // Сокровища появляются на возвышенности
                args.set(2, y + 10.0); // поднимаем на 10 блоков
                break;
                
            case "enemy":
                // Враги получают случайный разброс позиции
                double randomX = x + (Math.random() - 0.5) * 10;
                double randomY = y + (Math.random() - 0.5) * 10;
                args.set(1, randomX);
                args.set(2, randomY);
                break;
        }
        
        System.out.println("Spawning " + entityType + " at modified position");
    }
}
```

------------------------------------------------------------------------------------

Реализация случая с `NEW` (изменение аргументов конструктора) :  
Целевой класс :  
```java
public class ItemFactory {
    
    public Weapon createWeapon(String name, String material, int damage, float speed) {
        // Оригинальное создание оружия
        Weapon weapon = new Weapon(name, material, damage, speed);
        
        weapon.initialize();
        return weapon;
    }
}

class Weapon {
    private String name;
    private String material;
    private int damage;
    private float speed;
    
    public Weapon(String name, String material, int damage, float speed) {
        this.name = name;
        this.material = material;
        this.damage = damage;
        this.speed = speed;
    }
    
    public void initialize() {
        System.out.println("Weapon created: " + name + " (" + material + ") - " + 
                          damage + " damage, " + speed + " speed");
    }
}
```
Преобразующий класс :  
```java
@Mixin(ItemFactory.class)
public class ItemFactoryMixin {
    
    @ModifyArgs(method = "createWeapon(Ljava/lang/String;Ljava/lang/String;IF)LWeapon;",
                at = @At(value = "NEW", 
                        target = "(Ljava/lang/String;Ljava/lang/String;IF)V"))
    private void enhanceWeaponStats(Args args) {
        String name = args.get(0);
        String material = args.get(1);
        int damage = args.get(2);
        float speed = args.get(3);
        
        // Улучшаем все характеристики на основе материала
        String enhancedName = name;
        int enhancedDamage = damage;
        float enhancedSpeed = speed;
        
        switch (material.toLowerCase()) {
            case "iron":
                enhancedName = "Reinforced " + name;
                enhancedDamage = (int)(damage * 1.2f);
                enhancedSpeed = speed * 0.9f; // железо тяжелее
                break;
                
            case "steel":
                enhancedName = "Tempered " + name;
                enhancedDamage = (int)(damage * 1.4f);
                enhancedSpeed = speed * 1.0f; // сохраняем скорость
                break;
                
            case "mithril":
                enhancedName = "Legendary " + name;
                enhancedDamage = (int)(damage * 1.6f);
                enhancedSpeed = speed * 1.3f; // мифрил легкий и прочный
                break;
                
            default:
                enhancedName = "Common " + name;
                break;
        }
        
        // Устанавливаем улучшенные значения
        args.set(0, enhancedName);
        args.set(2, enhancedDamage);
        args.set(3, enhancedSpeed);
        
        System.out.println("Enhanced weapon creation: " + name + " -> " + enhancedName);
    }
}
```
Результат :  
При создании `new Weapon(name, material, damage, speed)` все аргументы будут модифицированы на основе типа материала.

------------------------------------------------------------------------------------

Реализация случая с валидацией и ограничением значений :  
```java
@Mixin(ServerConfig.class)
public class ServerConfigMixin {
    
    @ModifyArgs(method = "updateSettings(IIFF)V",
                at = @At(value = "INVOKE", 
                        target = "LServerConfig;applySettings(IIFF)V"))
    private void validateAndClampSettings(Args args) {
        int maxPlayers = args.get(0);
        int tickRate = args.get(1);
        float difficulty = args.get(2);
        float worldSize = args.get(3);
        
        // Применяем ограничения и валидацию
        int clampedMaxPlayers = Math.max(1, Math.min(maxPlayers, 100)); // 1-100 игроков
        int clampedTickRate = Math.max(10, Math.min(tickRate, 60));     // 10-60 TPS
        float clampedDifficulty = Math.max(0.1f, Math.min(difficulty, 5.0f)); // 0.1-5.0
        float clampedWorldSize = Math.max(1000f, Math.min(worldSize, 50000f)); // 1000-50000 блоков
        
        // Логируем изменения
        if (maxPlayers != clampedMaxPlayers) {
            System.out.println("Max players clamped: " + maxPlayers + " -> " + clampedMaxPlayers);
        }
        if (tickRate != clampedTickRate) {
            System.out.println("Tick rate clamped: " + tickRate + " -> " + clampedTickRate);
        }
        if (difficulty != clampedDifficulty) {
            System.out.println("Difficulty clamped: " + difficulty + " -> " + clampedDifficulty);
        }
        if (worldSize != clampedWorldSize) {
            System.out.println("World size clamped: " + worldSize + " -> " + clampedWorldSize);
        }
        
        // Устанавливаем проверенные значения
        args.set(0, clampedMaxPlayers);
        args.set(1, clampedTickRate);
        args.set(2, clampedDifficulty);
        args.set(3, clampedWorldSize);
    }
}
```

-----------------------------------------------------------------------------